In [1]:
# ============================================================
# PAMOJA NETWORK - ANOMALY DETECTION SYSTEM 
# ============================================================
   

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("🔍 PAMOJA NETWORK - ANOMALY DETECTION SYSTEM")
print("=" * 60)

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv('Loan_Default.csv')
print(f"\n✅ Loaded {len(df)} loan records")

REQUIRED_COLUMNS = ['loan_amount', 'rate_of_interest', 'income',
                    'Credit_Score', 'LTV', 'term']

missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(
        f"Loan_Default.csv is missing required column(s): {missing_cols}. "
        f"Available columns: {df.columns.tolist()}"
    )

# ============================================================
# 2. DERIVE THRESHOLDS & DEFAULTS FROM THE REAL DATA
# ============================================================


def derive_calibration(data):
    calib = {}
    for col in REQUIRED_COLUMNS:
        series = data[col].dropna()
        calib[col] = {
            'min': float(series.min()),
            'max': float(series.max()),
            'median': float(series.median()),
            'p1': float(series.quantile(0.01)),
            'p5': float(series.quantile(0.05)),
            'p95': float(series.quantile(0.95)),
            'p99': float(series.quantile(0.99)),
        }
    return calib


# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

def create_features(data, defaults, credit_range, stats=None):
    """
    defaults: dict of {col: fill_value} derived from real medians.
    credit_range: (min, max) derived from real data, used to normalize
                  Credit_Score into a 0-1 risk fraction (replaces the old
                  FICO-scale-assuming formula).
    stats: dict of {col: (mean, std)} for z-scores; computed from `data`
           itself if not supplied (only for the initial training fit).
    """
    df_copy = data.copy()

    for col in REQUIRED_COLUMNS:
        df_copy[col] = df_copy[col].fillna(defaults[col])

    # Financial ratios
    df_copy['debt_to_income'] = df_copy['loan_amount'] / (df_copy['income'] + 1)
    df_copy['income_to_loan'] = df_copy['income'] / (df_copy['loan_amount'] + 1)
    df_copy['ltv_ratio'] = df_copy['LTV'] / 100

    # Credit risk fraction: 0 = best score in the data, 1 = worst.
    # Calibrated to whatever scale Credit_Score is actually on.
    cmin, cmax = credit_range
    credit_risk_frac = (cmax - df_copy['Credit_Score']) / (cmax - cmin)
    credit_risk_frac = credit_risk_frac.clip(0, 1)

    df_copy['risk_score'] = credit_risk_frac * (df_copy['loan_amount'] / 100000)
    df_copy['payment_burden'] = df_copy['loan_amount'] / (df_copy['term'] * df_copy['income'] / 12 + 1)

    computed_stats = {}
    for col in ['loan_amount', 'income', 'Credit_Score', 'LTV']:
        if stats is not None:
            mean, std = stats[col]
        else:
            mean, std = df_copy[col].mean(), df_copy[col].std()
        computed_stats[col] = (mean, std)
        df_copy[f'{col}_zscore'] = (df_copy[col] - mean) / (std + 1)

    df_copy['combined_risk'] = (
        df_copy['loan_amount_zscore'] * 0.3 +
        credit_risk_frac * 0.3 +
        df_copy['ltv_ratio'] * 0.2 +
        df_copy['debt_to_income'] / 10 * 0.2
    )

    # Redundant *_zscore columns dropped from the ML feature set (v2 fix) -
    # they duplicate the raw columns after scaling and inflated how
    # "unusual" ordinary loans looked.
    feature_cols = [
        'loan_amount', 'rate_of_interest', 'income', 'Credit_Score', 'LTV',
        'debt_to_income', 'income_to_loan', 'ltv_ratio',
        'risk_score', 'payment_burden', 'combined_risk', 'term'
    ]

    return df_copy[feature_cols], computed_stats


# ============================================================
# 4. SPLIT FIRST, THEN CALIBRATE/FIT ON TRAIN ONLY
# ============================================================

df_train_raw, df_val_raw = train_test_split(df, test_size=0.2, random_state=42)
print(f"Training: {len(df_train_raw)} samples")
print(f"Validation: {len(df_val_raw)} samples")

calibration = derive_calibration(df_train_raw)
defaults = {col: calibration[col]['median'] for col in REQUIRED_COLUMNS}
credit_range = (calibration['Credit_Score']['min'], calibration['Credit_Score']['max'])

print("\n📐 Calibration derived from training data:")
for col in REQUIRED_COLUMNS:
    c = calibration[col]
    print(f"   {col}: p1={c['p1']:.1f}  p5={c['p5']:.1f}  median={c['median']:.1f}  "
          f"p95={c['p95']:.1f}  p99={c['p99']:.1f}")

imputer = SimpleImputer(strategy='median')
df_train_imputed = pd.DataFrame(
    imputer.fit_transform(df_train_raw[REQUIRED_COLUMNS]),
    columns=REQUIRED_COLUMNS, index=df_train_raw.index
)
df_val_imputed = pd.DataFrame(
    imputer.transform(df_val_raw[REQUIRED_COLUMNS]),
    columns=REQUIRED_COLUMNS, index=df_val_raw.index
)

X_train_feat, train_stats = create_features(df_train_imputed, defaults, credit_range)
X_val_feat, _ = create_features(df_val_imputed, defaults, credit_range, stats=train_stats)

print(f"\n✅ Created {X_train_feat.shape[1]} features (leak-free)")

# ============================================================
# 5. TRAIN ANOMALY DETECTOR
# ============================================================

print("\n" + "=" * 60)
print("TRAINING ANOMALY DETECTOR")
print("=" * 60)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_feat)
X_val_scaled = scaler.transform(X_val_feat)

# Fixed, conservative contamination (v2 fix) - the ML layer is a secondary,
# rare-pattern signal on top of the rule engine, not a competing classifier.
CONTAMINATION = 0.03

best_model = IsolationForest(
    contamination=CONTAMINATION,
    random_state=42,
    n_estimators=200,
    bootstrap=True
)
best_model.fit(X_train_scaled)

val_preds = best_model.predict(X_val_scaled)
val_flag_rate = (val_preds == -1).mean()
print(f"✅ Using fixed contamination={CONTAMINATION:.2%} "
      f"(validation flag rate: {val_flag_rate:.2%})")

# ============================================================
# 6. DETECTOR CLASS
# ============================================================

class AnomalyDetector:
    def __init__(self, model, scaler, feature_names, X_train_scaled,
                 train_stats, defaults, credit_range, calibration):
        self.model = model
        self.scaler = scaler
        self.feature_names = feature_names
        self.X_train_scaled = X_train_scaled
        self.train_stats = train_stats
        self.median_values = defaults
        self.credit_range = credit_range
        self.calibration = calibration  # per-column p1/p5/median/p95/p99 from real data
        self.threshold = self._calculate_threshold()

    def _calculate_threshold(self):
        scores = self.model.score_samples(self.X_train_scaled)
        return np.percentile(scores, 1)

    def detect(self, loan_data):
        violations = self._check_rules(loan_data)
        if violations:
            return {
                'alert': True,
                'severity': 'HIGH' if len(violations) >= 3 else 'MEDIUM',
                'reason': 'Rule violation detected',
                'details': violations
            }

        features = self._create_features(loan_data)
        features_df = pd.DataFrame([features])[self.feature_names]

        try:
            scaled = self.scaler.transform(features_df)
            pred = self.model.predict(scaled)[0]
            score = self.model.score_samples(scaled)[0]

            if pred == -1 or score < self.threshold:
                return {
                    'alert': True,
                    'severity': 'MEDIUM',
                    'reason': 'Unusual pattern detected',
                    'details': ['Loan deviates from normal patterns']
                }
        except Exception as e:
            print(f"⚠️  ML scoring failed, falling back to rule-only result: {e}")

        return {
            'alert': False,
            'severity': 'NONE',
            'reason': 'Loan appears normal',
            'details': []
        }

    def _get_value(self, data, key, alt_key=None):
        if alt_key is not None and alt_key in data:
            return data[alt_key]
        return data.get(key, self.median_values[key])

    def _create_features(self, data):
        loan = self._get_value(data, 'loan_amount', 'loan_Amount')
        interest = self._get_value(data, 'rate_of_interest')
        income = self._get_value(data, 'income')
        credit = self._get_value(data, 'Credit_Score', 'credit_score')
        ltv = self._get_value(data, 'LTV', 'ltv')
        term = self._get_value(data, 'term')

        debt_to_income = loan / (income + 1)
        income_to_loan = income / (loan + 1)
        ltv_ratio = ltv / 100
        cmin, cmax = self.credit_range
        credit_risk_frac = min(max((cmax - credit) / (cmax - cmin), 0), 1)
        risk_score = credit_risk_frac * (loan / 100000)
        payment_burden = loan / (term * income / 12 + 1)

        loan_mean, loan_std = self.train_stats['loan_amount']
        loan_amount_zscore = (loan - loan_mean) / (loan_std + 1)

        combined_risk = (
            loan_amount_zscore * 0.3 +
            credit_risk_frac * 0.3 +
            ltv_ratio * 0.2 +
            debt_to_income / 10 * 0.2
        )

        return {
            'loan_amount': loan, 'rate_of_interest': interest, 'income': income,
            'Credit_Score': credit, 'LTV': ltv,
            'debt_to_income': debt_to_income, 'income_to_loan': income_to_loan,
            'ltv_ratio': ltv_ratio, 'risk_score': risk_score,
            'payment_burden': payment_burden, 'combined_risk': combined_risk,
            'term': term
        }

    def _check_rules(self, data):
        """Rule-based anomaly checks, calibrated to the real data's own
        percentiles rather than hardcoded assumptions."""
        violations = []

        loan = self._get_value(data, 'loan_amount', 'loan_Amount')
        income = data.get('income', data.get('Income', None))
        credit = self._get_value(data, 'Credit_Score', 'credit_score')
        ltv = self._get_value(data, 'LTV', 'ltv')
        interest = self._get_value(data, 'rate_of_interest')

        c = self.calibration

        # Loan amount
        if loan >= c['loan_amount']['p99']:
            violations.append(f"Loan amount {loan:,.0f} is in the top 1% observed "
                               f"(p99: {c['loan_amount']['p99']:,.0f})")
        elif loan >= c['loan_amount']['p95']:
            violations.append(f"Loan amount {loan:,.0f} is in the top 5% observed "
                               f"(p95: {c['loan_amount']['p95']:,.0f})")

        # Income missing/zero + debt-to-income ratio
        if income is None or income <= 0:
            if loan > 0:
                violations.append("Income is missing or zero for a nonzero loan amount")
        else:
            debt_ratio = loan / income
            median_dti = c['loan_amount']['median'] / max(c['income']['median'], 1)
            if debt_ratio >= median_dti * 4:
                violations.append(f"Debt-to-income ratio {debt_ratio:.1f}x is far above "
                                   f"the typical {median_dti:.1f}x")
            elif debt_ratio >= median_dti * 2:
                violations.append(f"Debt-to-income ratio {debt_ratio:.1f}x is well above "
                                   f"the typical {median_dti:.1f}x")

        # Credit score - calibrated to this dataset's actual scale
        if credit <= c['Credit_Score']['p1']:
            violations.append(f"Credit score {credit:.0f} is in the bottom 1% observed "
                               f"(p1: {c['Credit_Score']['p1']:.0f})")
        elif credit <= c['Credit_Score']['p5']:
            violations.append(f"Credit score {credit:.0f} is in the bottom 5% observed "
                               f"(p5: {c['Credit_Score']['p5']:.0f})")

        # LTV - calibrated to this dataset's actual scale (can exceed 100%)
        if ltv >= c['LTV']['p99']:
            violations.append(f"LTV {ltv:.1f}% is in the top 1% observed "
                               f"(p99: {c['LTV']['p99']:.1f}%)")
        elif ltv >= c['LTV']['p95']:
            violations.append(f"LTV {ltv:.1f}% is in the top 5% observed "
                               f"(p95: {c['LTV']['p95']:.1f}%)")

        # Interest rate - calibrated to this dataset's actual scale
        if interest > 0:
            if interest >= c['rate_of_interest']['p99']:
                violations.append(f"Interest rate {interest:.2f}% is in the top 1% observed "
                                   f"(p99: {c['rate_of_interest']['p99']:.2f}%)")
            elif interest <= c['rate_of_interest']['p1']:
                violations.append(f"Interest rate {interest:.2f}% is in the bottom 1% observed "
                                   f"(p1: {c['rate_of_interest']['p1']:.2f}%)")

        return violations

    def get_summary(self, loan_data):
        features = self._create_features(loan_data)
        summary = {
            'loan_amount': features['loan_amount'],
            'rate_of_interest': features['rate_of_interest'],
            'income': features['income'],
            'Credit_Score': features['Credit_Score'],
            'LTV': features['LTV'],
            'debt_to_income': features['debt_to_income'],
            'income_to_loan': features['income_to_loan'],
            'risk_score': features['risk_score'],
            'payment_burden': features['payment_burden'],
            'combined_risk': features['combined_risk'],
            'rule_violations': self._check_rules(loan_data)
        }
        return summary


detector = AnomalyDetector(
    best_model, scaler, X_train_feat.columns.tolist(), X_train_scaled,
    train_stats, defaults, credit_range, calibration
)

# ============================================================
# 7. TEST THE SYSTEM (examples scaled to THIS dataset's real distribution)
# ============================================================

print("\n" + "=" * 60)
print("TESTING ANOMALY DETECTOR")
print("=" * 60)

med = {col: calibration[col]['median'] for col in REQUIRED_COLUMNS}

test_cases = [
    {
        'name': 'Typical loan (near medians)',
        'data': {'loan_amount': med['loan_amount'], 'rate_of_interest': med['rate_of_interest'],
                  'income': med['income'], 'Credit_Score': med['Credit_Score'],
                  'LTV': med['LTV'], 'term': 360}
    },
    {
        'name': 'Top-1% loan amount + bottom-1% credit + top-1% LTV',
        'data': {'loan_amount': calibration['loan_amount']['p99'],
                  'rate_of_interest': med['rate_of_interest'],
                  'income': calibration['income']['p5'],
                  'Credit_Score': calibration['Credit_Score']['p1'],
                  'LTV': calibration['LTV']['p99'], 'term': 360}
    },
    {
        'name': 'Undisclosed income',
        'data': {'loan_amount': med['loan_amount'], 'rate_of_interest': med['rate_of_interest'],
                  'income': 0, 'Credit_Score': med['Credit_Score'],
                  'LTV': med['LTV'], 'term': 360}
    },
]

for test in test_cases:
    result = detector.detect(test['data'])
    summary = detector.get_summary(test['data'])
    status = '🚨 ANOMALY' if result['alert'] else '✅ NORMAL'

    print(f"\n{test['name']}: {status}")
    print(f"  Debt-to-Income: {summary['debt_to_income']:.2f}x")
    print(f"  Income-to-Loan: {summary['income_to_loan']:.2%}")
    print(f"  Combined Risk: {summary['combined_risk']:.2f}")

    if result['alert']:
        print(f"  Severity: {result['severity']}")
        print(f"  Reason: {result['reason']}")
        for d in result['details']:
            print(f"    - {d}")

# ============================================================
# 8. SAVE MODELS
# ============================================================

print("\n" + "=" * 60)
print("💾 SAVING MODELS")
print("=" * 60)

os.makedirs('models', exist_ok=True)

joblib.dump(best_model, 'models/anomaly_detector.pkl')
joblib.dump(scaler, 'models/anomaly_scaler.pkl')
joblib.dump(X_train_feat.columns.tolist(), 'models/anomaly_features.pkl')
joblib.dump(imputer, 'models/anomaly_imputer.pkl')
joblib.dump(train_stats, 'models/anomaly_train_stats.pkl')
joblib.dump(calibration, 'models/anomaly_calibration.pkl')
joblib.dump(defaults, 'models/anomaly_defaults.pkl')
joblib.dump(credit_range, 'models/anomaly_credit_range.pkl')

print("✅ Models + calibration saved to 'models/' directory")

# ============================================================
# 9. INTERACTIVE TESTER
# ============================================================

print("\n" + "=" * 60)
print("🔍 INTERACTIVE ANOMALY DETECTION")
print("=" * 60)

QUIT_WORDS = {'quit', 'q', 'exit'}


def _read_field(prompt, default, caster):
    while True:
        raw = input(prompt).strip()
        if raw.lower() in QUIT_WORDS:
            return None, True
        if raw == '':
            return default, False
        try:
            return caster(raw), False
        except ValueError:
            print(f"❌ '{raw}' isn't a valid number, please try again (or type 'quit' to exit).")


def get_user_input():
    print("\n" + "-" * 50)
    print("📝 ENTER LOAN DETAILS:")
    print("(Press Enter to use defaults, type 'quit' to exit)")
    print("-" * 50)

    d = {}
    fields = [
        ('loan_amount', f"  Loan Amount [{med['loan_amount']:,.0f}]: ", med['loan_amount']),
        ('rate_of_interest', f"  Interest Rate (%) [{med['rate_of_interest']:.2f}]: ", med['rate_of_interest']),
        ('income', f"  Income [{med['income']:,.0f}]: ", med['income']),
        ('Credit_Score', f"  Credit Score [{med['Credit_Score']:.0f}]: ", med['Credit_Score']),
        ('LTV', f"  LTV Ratio (%) [{med['LTV']:.1f}]: ", med['LTV']),
    ]
    for key, prompt, default in fields:
        val, quit_now = _read_field(prompt, default, float)
        if quit_now:
            return None
        d[key] = val
    d['term'] = 360
    return d


def display_results(loan_data, result, summary):
    print("\n" + "=" * 60)
    print("📊 ANALYSIS RESULT")
    print("=" * 60)

    print("\n📋 LOAN DETAILS:")
    print(f"   • Loan Amount: {loan_data['loan_amount']:,.0f}")
    print(f"   • Interest Rate: {loan_data['rate_of_interest']:.2f}%")
    print(f"   • Income: {loan_data['income']:,.0f}")
    print(f"   • Credit Score: {loan_data['Credit_Score']:.0f}")
    print(f"   • LTV Ratio: {loan_data['LTV']:.1f}%")

    print("\n📊 CALCULATED METRICS:")
    print(f"   • Debt-to-Income Ratio: {summary['debt_to_income']:.2f}x")
    print(f"   • Income-to-Loan Ratio: {summary['income_to_loan']:.2%}")
    print(f"   • Combined Risk Index: {summary['combined_risk']:.2f}")

    print("\n🔍 DETECTION RESULT:")
    if result['alert']:
        print(f"   ⚠️  STATUS: ANOMALY DETECTED!")
        print(f"   📊 Severity: {result['severity']}")
        print(f"   📝 Reason: {result['reason']}")
        if result['details']:
            print(f"\n   📋 Issues Found:")
            for detail in result['details']:
                print(f"      • {detail}")
        if result['severity'] == 'HIGH':
            print(f"\n   🔴 RISK LEVEL: HIGH")
            print(f"   💡 Recommendation: REJECT this loan")
        elif result['severity'] == 'MEDIUM':
            print(f"\n   🟡 RISK LEVEL: MEDIUM")
            print(f"   💡 Recommendation: Review carefully")
    else:
        print(f"   ✅ STATUS: NO ANOMALY")
        print(f"   💚 {result['reason']}")
        print(f"\n   🟢 RISK LEVEL: LOW")
        print(f"   💡 Recommendation: APPROVE with standard terms")

    print("=" * 60)


if __name__ == '__main__':
    while True:
        loan_data = get_user_input()
        if loan_data is None:
            print("\n👋 Thank you for using Pamoja Network!")
            break

        cmin, cmax = credit_range
        if loan_data['Credit_Score'] < cmin or loan_data['Credit_Score'] > cmax:
            print(f"⚠️  Credit score must be between {cmin:.0f} and {cmax:.0f}")
            continue

        result = detector.detect(loan_data)
        summary = detector.get_summary(loan_data)
        display_results(loan_data, result, summary)

        another = input("\n🔁 Test another loan? (y/n): ").lower()
        if another != 'y':
            print("\n👋 Thank you for using Pamoja Network!")
            break

    print("\n" + "=" * 60)
    print("🏦 Pamoja Network - Smart Chama Management")
    print("=" * 60)

🔍 PAMOJA NETWORK - ANOMALY DETECTION SYSTEM

✅ Loaded 5488 loan records
Training: 4390 samples
Validation: 1098 samples

📐 Calibration derived from training data:
   loan_amount: p1=66500.0  p5=106500.0  median=296500.0  p95=666500.0  p99=817600.0
   rate_of_interest: p1=2.9  p5=3.2  median=4.0  p95=5.0  p99=5.4
   income: p1=1033.2  p5=2040.0  median=5820.0  p95=15000.0  p99=25653.6
   Credit_Score: p1=504.0  p5=522.0  median=698.0  p95=881.0  p99=896.0
   LTV: p1=19.9  p5=36.8  median=75.1  p95=98.8  p99=102.6
   term: p1=156.0  p5=180.0  median=360.0  p95=360.0  p99=360.0

✅ Created 12 features (leak-free)

TRAINING ANOMALY DETECTOR
✅ Using fixed contamination=3.00% (validation flag rate: 3.01%)

TESTING ANOMALY DETECTOR

Typical loan (near medians): ✅ NORMAL
  Debt-to-Income: 50.94x
  Income-to-Loan: 1.96%
  Combined Risk: 1.26

Top-1% loan amount + bottom-1% credit + top-1% LTV: 🚨 ANOMALY
  Debt-to-Income: 400.59x
  Income-to-Loan: 0.25%
  Combined Risk: 9.27
  Severity: HIGH
  Re

  Loan Amount [296,500]:  l


❌ 'l' isn't a valid number, please try again (or type 'quit' to exit).


  Loan Amount [296,500]:  5699
  Interest Rate (%) [3.99]:  6.0
  Income [5,820]:  3000
  Credit Score [698]:  6798
  LTV Ratio (%) [75.1]:  899


⚠️  Credit score must be between 500 and 900

--------------------------------------------------
📝 ENTER LOAN DETAILS:
(Press Enter to use defaults, type 'quit' to exit)
--------------------------------------------------


  Loan Amount [296,500]:  quit



👋 Thank you for using Pamoja Network!

🏦 Pamoja Network - Smart Chama Management
